# Text Data Preparation

Builds the controlled text-preparation layer from `01_data/interim/ev_bus_text_corpus.csv`.

This notebook does **not** calculate final sentiment, lifecycle findings, or topic models. It preserves source values, standardises controlled entities, creates separate sentiment/topic text fields, flags quality risks, and saves all derived tables outside notebook state.

Governance sources: `PHASE1_DECISIONS.md`, `DATA_DICTIONARY.md`, and `METHODOLOGY.md`. Synthetic data is not used.

In [1]:
# Package installation cell.
# The repository is pinned for Python 3.11. Set True only when preparing a fresh environment.
INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"])
else:
    print("Using the active environment. Set INSTALL_PACKAGES=True for a fresh environment.")

In [2]:
from __future__ import annotations

import hashlib
import json
import random
import re
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "02_notebooks" else Path.cwd()
INPUT_PATH = PROJECT_ROOT / "01_data/interim/ev_bus_text_corpus.csv"
PROCESSED_PATH = PROJECT_ROOT / "01_data/processed/ev_bus_text_corpus_processed.csv"
OUTPUT_DIR = PROJECT_ROOT / "04_outputs/text"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Input:", INPUT_PATH)
print("Output directory:", OUTPUT_DIR)

In [3]:
# Load the approved interim file without modifying it.
source_df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

EXPECTED_COLUMNS = ["Text", "source", "brand", "entity_type", "url", "date", "camp"]
if list(source_df.columns) != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected input schema: {list(source_df.columns)}")
if len(source_df) != 548:
    raise ValueError(f"Expected 548 input rows; found {len(source_df)}")

source_df.head()

## Controlled mappings and derived provenance

The input has no separate platform, provenance, or original-record-ID field. `platform` is derived from the preserved `source`. Provenance uses the Phase 1 audit lineage:

- unlinked YouTube rows are labelled `direct_platform_export_unlinked`;
- non-Instagram rows with source-linked secondary excerpts are labelled `ai_compiled_secondary`;
- Instagram rows are labelled `mixed_or_uncertain` because the merged corpus does not distinguish direct salvaged comments from AI-compiled Instagram excerpts.

No missing URL, date, or ID is fabricated.

In [4]:
ENTITY_MAP = {
    "general": ("General/Industry", "General/Industry", "not_applicable"),
    "industry": ("General/Industry", "General/Industry", "not_applicable"),
    "tata": ("Tata Motors", "OEM", "legacy"),
    "tata motors": ("Tata Motors", "OEM", "legacy"),
    "jbm": ("JBM Auto", "OEM", "legacy"),
    "jbm auto": ("JBM Auto", "OEM", "legacy"),
    "vecv": ("VECV / Eicher", "OEM", "legacy"),
    "ve commercial vehicles": ("VECV / Eicher", "OEM", "legacy"),
    "eicher": ("VECV / Eicher", "OEM", "legacy"),
    "olectra": ("Olectra Greentech", "OEM", "new_age"),
    "olectra greentech": ("Olectra Greentech", "OEM", "new_age"),
    "eka": ("EKA Mobility", "OEM", "new_age"),
    "eka mobility": ("EKA Mobility", "OEM", "new_age"),
    "pmi": ("PMI Electro", "OEM", "new_age"),
    "pmi electro": ("PMI Electro", "OEM", "new_age"),
    "pmi electro mobility": ("PMI Electro", "OEM", "new_age"),
    "switch": ("Switch Mobility", "OEM", "new_age"),
    "switch mobility": ("Switch Mobility", "OEM", "new_age"),
    "nuego": ("NueGo", "Operator", "not_applicable"),
    "nuego india": ("NueGo", "Operator", "not_applicable"),
    "anthony travels": ("Anthony Travels", "Operator", "not_applicable"),
    "anthony travels consortium": ("Anthony Travels", "Operator", "not_applicable"),
}

def normalise_label(value: str) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

def map_entity(value: str) -> tuple[str, str, str]:
    key = normalise_label(value)
    return ENTITY_MAP.get(key, ("General/Industry", "General/Industry", "not_applicable"))

def derive_provenance(row: pd.Series) -> str:
    source = normalise_label(row["source"])
    if source == "youtube" and not row["url"].strip() and not row["date"].strip():
        return "direct_platform_export_unlinked"
    if source == "instagram":
        return "mixed_or_uncertain"
    return "ai_compiled_secondary"

df = source_df.copy()

# Preserve every original field under its original name and create explicit lineage fields.
df["source_file"] = "ev_bus_text_corpus.csv"
df["source_text_original"] = df["Text"]
df["raw_text"] = df["Text"]
df["entity_name_original"] = df["brand"]
mapped = df["brand"].map(map_entity)
df[["entity_name_canonical", "entity_type_controlled", "oem_group"]] = pd.DataFrame(mapped.tolist(), index=df.index)
df["entity_type_original"] = df["entity_type"]
df["entity_type"] = df["entity_type_controlled"]
df.drop(columns=["entity_type_controlled"], inplace=True)
df["platform"] = df["source"].where(df["source"].str.strip().ne(""), "Unknown")
df["provenance"] = df.apply(derive_provenance, axis=1)

# Stable ID: original lineage values + occurrence number, so exact duplicates remain distinct.
id_basis = df[EXPECTED_COLUMNS].astype(str).agg("".join, axis=1)
occurrence = id_basis.groupby(id_basis).cumcount() + 1
df["comment_id"] = [
    "c_" + hashlib.sha256(f"{basis}{occ}".encode("utf-8")).hexdigest()[:20]
    for basis, occ in zip(id_basis, occurrence)
]

assert df["comment_id"].is_unique
assert not ((df["entity_name_canonical"].isin(["NueGo", "Anthony Travels"])) & (df["entity_type"] == "OEM")).any()
assert not ((df["entity_type"] != "OEM") & (df["oem_group"] != "not_applicable")).any()

df[["comment_id", "entity_name_original", "entity_name_canonical", "entity_type", "oem_group", "platform", "provenance"]].head()

In [5]:
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)

def make_sentiment_text(text: str) -> str:
    text = URL_RE.sub(" ", str(text))
    # Collapse unnecessary horizontal whitespace while retaining meaningful line breaks.
    return chr(10).join(" ".join(line.split()) for line in text.splitlines()).strip()

df["sentiment_text"] = df["raw_text"].map(make_sentiment_text)

# Negations, punctuation, stopwords, capitalization, and emojis are intentionally retained.
assert len(df) == len(source_df)
df[["raw_text", "sentiment_text"]].head()

In [6]:
# Topic-model preprocessing configuration.
# Stopwords are explicit for reproducibility. Domain terms are protected from stopword removal.

DOMAIN_TERMS = {
    "ev", "electric", "ebus", "ebuses", "bus", "buses", "battery", "charging", "charger",
    "range", "depot", "fleet", "tender", "procurement", "operator", "oem", "mobility",
    "service", "maintenance", "breakdown", "refund", "route", "fare", "passenger", "driver",
    "tata", "jbm", "vecv", "eicher", "olectra", "eka", "pmi", "switch", "nuego",
}
STOPWORDS = set("a about above after again against all am an and any are as at be because been before being below between both but by can could did do does doing down during each few for from further had has have having he her here hers herself him himself his how i if in into is it its itself just me more most my myself no nor not now of off on once only or other our ours ourselves out over own same she should so some such than that the their theirs them themselves then there these they this those through to too under until up very was we were what when where which while who whom why will with you your yours yourself yourselves".split()) - DOMAIN_TERMS

HINGLISH_TERMS = {
    "aap", "aapko", "acha", "achha", "agar", "aur", "bahut", "bhai", "bhi", "hai", "hain",
    "ham", "hum", "ka", "kar", "karo", "ke", "ki", "ko", "kya", "lekin", "mat",
    "mein", "mera", "nahi", "nhi", "paisa", "par", "raha", "rahi", "sahi", "se", "sirf",
    "tha", "toh", "wala", "wali", "yaar", "ye", "yeh",
}

IRREGULAR_LEMMAS = {
    "buses": "bus", "ebuses": "ebus", "batteries": "battery", "cities": "city",
    "companies": "company", "vehicles": "vehicle", "services": "service", "drivers": "driver",
    "passengers": "passenger", "routes": "route", "fares": "fare", "chargers": "charger",
    "charging": "charge", "charged": "charge", "breakdowns": "breakdown", "issues": "issue",
    "orders": "order", "tenders": "tender", "comments": "comment", "posts": "post",
    "better": "good", "best": "good", "worse": "bad", "worst": "bad",
}

def deterministic_lemma(token: str) -> str:
    """Conservative, corpus-independent lemma rules (version 1)."""
    if token in DOMAIN_TERMS:
        return IRREGULAR_LEMMAS.get(token, token)
    if token in IRREGULAR_LEMMAS:
        return IRREGULAR_LEMMAS[token]
    if len(token) > 5 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 5 and token.endswith("ing") and not token.endswith("thing"):
        base = token[:-3]
        if len(base) >= 3 and base[-1:] == base[-2:-1]:
            base = base[:-1]
        return base
    if len(token) > 4 and token.endswith("ed"):
        return token[:-2]
    if len(token) > 4 and token.endswith("s") and not token.endswith(("ss", "us", "is")):
        return token[:-1]
    return token

def make_topic_text(text: str) -> tuple[str, str]:
    text = URL_RE.sub(" ", str(text).lower())
    # Punctuation and numbers are replaced with spaces; Unicode alphabetic characters are retained.
    text = "".join(ch if ch.isalpha() or ch.isspace() else " " for ch in text)
    tokens = [t for t in re.findall(r"[^\W\d_]+", text, flags=re.UNICODE) if len(t) > 1]
    hinglish = sorted(set(t for t in tokens if t in HINGLISH_TERMS))
    kept = [deterministic_lemma(t) for t in tokens if t not in STOPWORDS]
    return " ".join(kept), ";".join(hinglish)

topic_parts = df["raw_text"].map(make_topic_text)
df["topic_model_text"] = topic_parts.map(lambda x: x[0])
df["hinglish_terms_detected"] = topic_parts.map(lambda x: x[1])
df["topic_preprocessing_version"] = "deterministic_domain_lemma_v1"

df[["raw_text", "topic_model_text", "hinglish_terms_detected"]].head()

In [7]:
# Quality flags. Most flags trigger review, not exclusion.
def normalise_for_duplicate(text: str) -> str:
    return re.sub(r"\s+", " ", str(text).casefold()).strip()

df["duplicate_key"] = df["raw_text"].map(normalise_for_duplicate)
duplicate_mask = df["duplicate_key"].duplicated(keep=False) & df["duplicate_key"].ne("")

# Deterministic standard-library near-duplicate scan.
near_duplicate_mask = pd.Series(False, index=df.index)
near_duplicate_partner = pd.Series("", index=df.index, dtype=str)
texts = df["duplicate_key"].tolist()
best_match = [(None, 0.0) for _ in texts]
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        if texts[i] == texts[j] or not texts[i] or not texts[j]:
            continue
        # Avoid expensive comparisons for texts with clearly different lengths.
        ratio = min(len(texts[i]), len(texts[j])) / max(len(texts[i]), len(texts[j]))
        if ratio < 0.85:
            continue
        score = SequenceMatcher(None, texts[i], texts[j], autojunk=False).ratio()
        if score >= 0.92:
            near_duplicate_mask.iloc[i] = True
            near_duplicate_mask.iloc[j] = True
            if score > best_match[i][1]: best_match[i] = (j, score)
            if score > best_match[j][1]: best_match[j] = (i, score)
for i, (j, score) in enumerate(best_match):
    if j is not None:
        near_duplicate_partner.iloc[i] = f"{df.iloc[j]['comment_id']}|{score:.4f}"

alpha_num = df["raw_text"].map(lambda x: "".join(ch for ch in str(x) if ch.isalnum()))
word_count = df["raw_text"].map(lambda x: len(str(x).split()))
very_short_mask = (alpha_num.str.len() < 15) | (word_count < 3)
emoji_only_mask = df["raw_text"].map(lambda x: bool(str(x).strip()) and not any(ch.isalnum() for ch in str(x)))
blank_mask = df["raw_text"].str.strip().eq("")
url_only_mask = df["sentiment_text"].str.strip().eq("") & df["raw_text"].str.contains(URL_RE, na=False)

PROMO_PHRASES = ["proud", "thrilled", "excited", "introducing", "launched", "launches", "book now", "experience the future", "milestone", "official", "redefining", "revolutionizing", "revolutionising", "join us", "celebrate", "celebrating"]
possible_promo_mask = df["raw_text"].map(lambda x: any(phrase in str(x).casefold() for phrase in PROMO_PHRASES))
possible_brand_authored_mask = possible_promo_mask & df["platform"].str.casefold().isin(["instagram", "linkedin", "facebook"])

EV_CONTEXT_TERMS = {"bus", "buses", "ebus", "electric", "ev", "battery", "charging", "charger", "mobility", "transport", "tata", "jbm", "olectra", "nuego", "pmi", "eka", "switch", "vecv", "eicher", "fleet", "depot", "route", "passenger", "tender"}
OFF_TOPIC_TERMS = {"laptop", "smartphone", "cricket", "religion", "election", "recipe", "gaming", "stocks"}
def lower_tokens(text: str) -> set[str]:
    return set(re.findall(r"[a-z]+", str(text).casefold()))
token_sets = df["raw_text"].map(lower_tokens)
possible_off_topic_mask = token_sets.map(lambda tokens: bool(tokens & OFF_TOPIC_TERMS) and not bool(tokens & EV_CONTEXT_TERMS))

missing_source_mask = df["source"].str.strip().eq("") | (df["url"].str.strip().eq("") & df["date"].str.strip().eq(""))
hinglish_mask = df["hinglish_terms_detected"].str.strip().ne("")

flag_masks = {
    "duplicate_text": duplicate_mask,
    "near_duplicate_text": near_duplicate_mask,
    "very_short_text": very_short_mask,
    "emoji_only_text": emoji_only_mask,
    "possible_promotional_text": possible_promo_mask,
    "possible_brand_authored_content": possible_brand_authored_mask,
    "possible_off_topic_content": possible_off_topic_mask,
    "missing_source_information": missing_source_mask,
    "hinglish_or_code_switched": hinglish_mask,
}

def join_flags(index: int) -> str:
    flags = [name for name, mask in flag_masks.items() if bool(mask.iloc[index])]
    return ";".join(flags) if flags else "none"

df["quality_flag"] = [join_flags(i) for i in range(len(df))]
df["near_duplicate_partner"] = near_duplicate_partner

# Only clearly unusable text is automatically excluded.
exclusion_masks = {
    "blank_text": blank_mask,
    "url_only_text": url_only_mask,
    "emoji_only_text": emoji_only_mask,
}

def join_exclusions(index: int) -> str:
    reasons = [name for name, mask in exclusion_masks.items() if bool(mask.iloc[index])]
    return ";".join(reasons)

df["exclusion_reason"] = [join_exclusions(i) for i in range(len(df))]
df["analysis_eligible"] = df["exclusion_reason"].eq("")

pd.DataFrame({name: [int(mask.sum())] for name, mask in flag_masks.items()})

In [8]:
# Arrange lineage, controlled fields, text fields, quality fields, then untouched source columns.
OUTPUT_COLUMNS = [
    "comment_id", "source_file", "platform", "source", "url", "date", "provenance",
    "entity_name_original", "entity_name_canonical", "entity_type", "oem_group",
    "raw_text", "sentiment_text", "topic_model_text", "hinglish_terms_detected",
    "topic_preprocessing_version", "quality_flag", "exclusion_reason", "analysis_eligible",
    "near_duplicate_partner", "source_text_original", "Text", "brand", "entity_type_original", "camp",
]
processed = df[OUTPUT_COLUMNS].copy()

assert len(processed) == len(source_df)
assert processed["raw_text"].equals(source_df["Text"])
assert processed["entity_name_original"].equals(source_df["brand"])
assert processed["source"].equals(source_df["source"])
assert processed["url"].equals(source_df["url"])
assert processed["date"].equals(source_df["date"])

processed.to_csv(PROCESSED_PATH, index=False)
print(f"Saved {len(processed):,} rows to {PROCESSED_PATH}")

In [9]:
# Derived quality and distribution tables.
def distribution(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    result = frame.groupby(columns, dropna=False).size().reset_index(name="row_count")
    result["percent_of_rows"] = (result["row_count"] / len(frame) * 100).round(2)
    return result.sort_values("row_count", ascending=False).reset_index(drop=True)

entity_distribution = distribution(processed, ["entity_name_canonical", "entity_type", "oem_group"])
platform_distribution = distribution(processed, ["platform"])
provenance_distribution = distribution(processed, ["provenance"])
oem_group_distribution = distribution(processed, ["oem_group"])

quality_rows = [
    {"section": "row_flow", "metric": "input_rows", "count": len(source_df), "percent_of_input": 100.0, "definition": "Rows in approved interim input"},
    {"section": "row_flow", "metric": "output_rows", "count": len(processed), "percent_of_input": round(len(processed)/len(source_df)*100, 2), "definition": "Rows retained in processed corpus, including flagged/excluded records"},
    {"section": "row_flow", "metric": "analysis_eligible_rows", "count": int(processed["analysis_eligible"].sum()), "percent_of_input": round(processed["analysis_eligible"].mean()*100, 2), "definition": "Rows not automatically excluded"},
    {"section": "row_flow", "metric": "automatically_excluded_rows", "count": int((~processed["analysis_eligible"]).sum()), "percent_of_input": round((~processed["analysis_eligible"]).mean()*100, 2), "definition": "Rows excluded only for blank, URL-only, or emoji-only text"},
]
for name, mask in flag_masks.items():
    quality_rows.append({"section": "quality_flag", "metric": name, "count": int(mask.sum()), "percent_of_input": round(mask.mean()*100, 2), "definition": "Non-exclusive flag; rows may appear in multiple metrics"})
for name, mask in exclusion_masks.items():
    quality_rows.append({"section": "exclusion_reason", "metric": name, "count": int(mask.sum()), "percent_of_input": round(mask.mean()*100, 2), "definition": "Automatic exclusion reason"})
for col in ["source", "url", "date", "entity_name_original", "raw_text"]:
    missing = int(processed[col].astype(str).str.strip().eq("").sum())
    quality_rows.append({"section": "missing_value", "metric": f"missing_{col}", "count": missing, "percent_of_input": round(missing/len(processed)*100, 2), "definition": f"Blank values in {col}"})

data_quality_summary = pd.DataFrame(quality_rows)
exclusion_log_columns = [
    "comment_id", "source_file", "platform", "source", "url", "date", "provenance",
    "entity_name_original", "entity_name_canonical", "entity_type", "oem_group",
    "raw_text", "quality_flag", "exclusion_reason",
]
exclusion_log = processed.loc[~processed["analysis_eligible"], exclusion_log_columns].copy()

data_quality_summary.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)
entity_distribution.to_csv(OUTPUT_DIR / "entity_distribution.csv", index=False)
platform_distribution.to_csv(OUTPUT_DIR / "platform_distribution.csv", index=False)
provenance_distribution.to_csv(OUTPUT_DIR / "provenance_distribution.csv", index=False)
exclusion_log.to_csv(OUTPUT_DIR / "exclusion_log.csv", index=False)

display(data_quality_summary)
display(entity_distribution)
display(platform_distribution)
display(provenance_distribution)

In [10]:
# Generate the preprocessing report from the saved results.
flag_counts = {name: int(mask.sum()) for name, mask in flag_masks.items()}
exclusion_counts = {name: int(mask.sum()) for name, mask in exclusion_masks.items()}
missing_counts = {
    col: int(processed[col].astype(str).str.strip().eq("").sum())
    for col in ["source", "url", "date", "entity_name_original", "raw_text"]
}

def markdown_table(frame: pd.DataFrame) -> str:
    headers = [str(c) for c in frame.columns]
    def safe(value):
        return str(value).replace("|", "\|").replace(chr(10), " ")
    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    lines.extend("| " + " | ".join(safe(v) for v in row) + " |" for row in frame.itertuples(index=False, name=None))
    return chr(10).join(lines)

report = f"""# Text Preprocessing Report

## Scope

This layer prepares the approved mixed-provenance 548-row text corpus. It does not calculate final sentiment, lifecycle findings, or topic models. No synthetic data was used.

## Row flow

- Input rows: **{len(source_df):,}**
- Processed output rows: **{len(processed):,}**
- Analysis-eligible rows: **{int(processed["analysis_eligible"].sum()):,}**
- Automatically excluded rows: **{int((~processed["analysis_eligible"]).sum()):,}**

All input rows remain in the processed file for lineage. `analysis_eligible=False` identifies clearly unusable rows; only those rows are also written to the exclusion log.

## Duplicate and quality flags

| Flag | Rows |
|---|---:|
{chr(10).join(f'| `{name}` | {count:,} |' for name, count in flag_counts.items())}

Exact duplicates use case-folded, whitespace-normalised raw text. Near duplicates use deterministic standard-library sequence similarity with a 0.92 threshold and a minimum length ratio of 0.85. These flags are review signals and do not automatically remove rows.

## Exclusions by reason

| Reason | Rows |
|---|---:|
{chr(10).join(f'| `{name}` | {count:,} |' for name, count in exclusion_counts.items())}

Automatic exclusion is deliberately limited to blank, URL-only, and emoji-only text.

## Entity distribution

{markdown_table(entity_distribution)}

## OEM-group distribution

{markdown_table(oem_group_distribution)}

Anthony Travels and NueGo are controlled as operators and have `oem_group=not_applicable`; they are never included as OEMs.

## Platform distribution

{markdown_table(platform_distribution)}

The input has no independent platform field, so `platform` is derived from the preserved `source` value.

## Provenance distribution

{markdown_table(provenance_distribution)}

The input has no row-level source-file or provenance field. Provenance is derived conservatively from the approved Phase 1 lineage. Instagram rows are `mixed_or_uncertain` because direct salvaged comments cannot be distinguished reliably from AI-compiled excerpts in the merged file.

## Missing-value summary

| Field | Missing rows |
|---|---:|
{chr(10).join(f'| `{name}` | {count:,} |' for name, count in missing_counts.items())}

Missing URL/date values are retained and flagged; no values are fabricated.

## Text representations

- `raw_text` is an unchanged copy of input `Text`.
- `sentiment_text` removes URLs and unnecessary whitespace only. Negations, stopwords, meaningful punctuation, capitalization, and emojis are retained.
- `topic_model_text` lowercases, removes URLs/punctuation/numbers and English stopwords, protects EV-bus domain terms, and applies conservative deterministic lemma rules (`deterministic_domain_lemma_v1`).
- Detected Hinglish tokens are retained in topic text unless they are independently removed by an explicit future rule, and are also recorded in `hinglish_terms_detected` for audit.

## Corpus limitations

- The corpus mixes direct platform exports with AI/Manus-compiled secondary excerpts.
- Most rows lack record-level IDs; stable IDs here are deterministic derived identifiers, not platform IDs.
- URL and date coverage is incomplete, limiting verification and time-based analysis.
- YouTube and General/Industry content dominate the sample; OEM and operator samples are much smaller.
- Sampling was not random or balanced across platforms, entities, time, or customer lifecycle.
- Instagram provenance is ambiguous in the merged file, and promotional/brand-authored material may remain.
- Hinglish and code-switched detection is a transparent term-list flag, not a full language classifier.
- Promotional, brand-authored, off-topic, short, duplicate, and near-duplicate rules are heuristics and require manual review.
- Topic preprocessing uses a conservative corpus-independent lemma rule set to avoid hidden downloads; it is less linguistically complete than a fully versioned POS-aware WordNet pipeline.
- This preparation layer does not validate sentiment accuracy, lifecycle labels, topic stability, or representativeness.

## Reproducibility

- All notebook paths are relative to the project root.
- Random seed: `{SEED}`.
- The approved interim input is read-only.
- Every derived CSV and this report are saved outside notebook state.
- No synthetic data, network browsing, final sentiment calculation, lifecycle classification, or topic modelling is performed.
"""

(OUTPUT_DIR / "preprocessing_report.md").write_text(report, encoding="utf-8")
print(OUTPUT_DIR / "preprocessing_report.md")

In [11]:
# Final integrity checks.
expected_files = [
    PROCESSED_PATH,
    OUTPUT_DIR / "data_quality_summary.csv",
    OUTPUT_DIR / "entity_distribution.csv",
    OUTPUT_DIR / "platform_distribution.csv",
    OUTPUT_DIR / "provenance_distribution.csv",
    OUTPUT_DIR / "exclusion_log.csv",
    OUTPUT_DIR / "preprocessing_report.md",
]
for path in expected_files:
    assert path.exists(), f"Missing output: {path}"

reloaded = pd.read_csv(PROCESSED_PATH, dtype=str, keep_default_na=False)
assert len(reloaded) == 548
assert reloaded["comment_id"].is_unique
assert reloaded["raw_text"].tolist() == source_df["Text"].tolist()
assert not ((reloaded["entity_name_canonical"].isin(["NueGo", "Anthony Travels"])) & (reloaded["entity_type"] == "OEM")).any()
assert not ((reloaded["entity_type"] != "OEM") & (reloaded["oem_group"] != "not_applicable")).any()

print("Integrity checks passed.")
print("Created:")
for path in expected_files:
    print(" -", path.relative_to(PROJECT_ROOT))

## Targeted final data-quality patch (v2)

This patch starts from the structurally approved v1 processed CSV. It preserves all 548 rows and all existing columns, builds graph-based near-duplicate clusters from `near_duplicate_partner`, adds separate sentiment/topic eligibility, reviews speaker role, normalises platforms, and regenerates topic text without aggressive suffix stemming.

It does not run lifecycle classification, VADER sentiment, or LDA/topic modelling.

In [12]:
# Targeted v2 patch: no lifecycle, sentiment scoring, or topic modelling.
from __future__ import annotations

import hashlib
import json
import random
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

PATCH_SEED = 42
random.seed(PATCH_SEED)
np.random.seed(PATCH_SEED)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "02_notebooks" else Path.cwd()
V1_PATH = PROJECT_ROOT / "01_data/processed/ev_bus_text_corpus_processed.csv"
V2_PATH = PROJECT_ROOT / "01_data/processed/ev_bus_text_corpus_processed_v2.csv"
PATCH_OUTPUT_DIR = PROJECT_ROOT / "04_outputs/text"
REVIEW_PATH = PATCH_OUTPUT_DIR / "manual_quality_review.csv"
SUMMARY_PATH = PATCH_OUTPUT_DIR / "final_eligibility_summary.csv"
REPORT_PATH = PATCH_OUTPUT_DIR / "preprocessing_patch_report.md"
PATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

v1 = pd.read_csv(V1_PATH, dtype=str, keep_default_na=False)
if len(v1) != 548:
    raise ValueError(f"Expected 548 approved rows; found {len(v1)}")
if not v1["comment_id"].is_unique:
    raise ValueError("Approved comment_id is not unique")

v2 = v1.copy()
v1_columns = list(v1.columns)

# Platform normalisation: preserve both the existing platform and an explicit original copy.
v2["platform_original"] = v2["platform"]
PLATFORM_MAP = {
    "youtube": "YouTube", "youtube comments": "YouTube",
    "instagram": "Instagram",
    "reddit": "Reddit",
    "facebook": "Facebook", "facebook group": "Facebook",
    "linkedin": "LinkedIn", "quora/linkedin": "LinkedIn",
    "forum": "Forum", "team-bhp": "Forum", "quora": "Forum",
    "mouthshut": "Review Platform", "news/ratings": "Review Platform",
    "glassdoor/forum": "Review Platform", "ev care": "Review Platform", "ambitionbox": "Review Platform",
    "news": "News/Media", "news article": "News/Media",
}
v2["platform_canonical"] = v2["platform_original"].map(
    lambda x: PLATFORM_MAP.get(str(x).strip().casefold(), "Mixed/Unknown")
)

# Transparent flags inherited from v1.
def has_flag(series: pd.Series, flag: str) -> pd.Series:
    return series.str.split(";").map(lambda values: flag in values)

flag_duplicate = has_flag(v2["quality_flag"], "duplicate_text")
flag_near = has_flag(v2["quality_flag"], "near_duplicate_text")
flag_short = has_flag(v2["quality_flag"], "very_short_text")
flag_promo = has_flag(v2["quality_flag"], "possible_promotional_text")
flag_brand_v1 = has_flag(v2["quality_flag"], "possible_brand_authored_content")
flag_off_topic = has_flag(v2["quality_flag"], "possible_off_topic_content")
flag_hinglish = has_flag(v2["quality_flag"], "hinglish_or_code_switched")

text_cf = v2["raw_text"].str.casefold()
word_count = v2["raw_text"].map(lambda text: len(str(text).split()))

# Speaker role: conservative, with unknown where Instagram provenance is ambiguous.
CREATOR_PATTERNS = [
    "your anchor here", "topics would you want me to cover", "thanks for watching",
    "thank you for watching", "welcome back to my channel", "welcome back to our channel",
    "we are proud to announce", "we're proud to announce", "we are thrilled to announce",
    "we're thrilled to announce", "introducing our",
]
creator_explicit = text_cf.map(lambda text: any(pattern in text for pattern in CREATOR_PATTERNS))
user_voice_markers = [
    "user says:", "i ", "i'm", "i am", "my ", "me ", "please refund", "my experience",
    "disappointed", "worst service", "reached out", "waiting for a response",
]
instagram_user_voice = (v2["platform_canonical"] == "Instagram") & text_cf.map(
    lambda text: any(marker in text for marker in user_voice_markers)
)
instagram_oem_caption = (
    (v2["platform_canonical"] == "Instagram")
    & (v2["entity_type"] == "OEM")
    & ~instagram_user_voice
    & ~text_cf.str.startswith("user says:")
)
creator_or_brand_flag = creator_explicit | flag_brand_v1 | instagram_oem_caption

audience_platform = v2["platform_canonical"].isin(
    ["YouTube", "Reddit", "Facebook", "Forum", "Review Platform"]
)
media_platform = v2["platform_canonical"].isin(["LinkedIn", "News/Media"])

v2["speaker_role"] = "unknown"
v2.loc[audience_platform, "speaker_role"] = "audience_user"
v2.loc[media_platform, "speaker_role"] = "media_or_research"
v2.loc[instagram_user_voice, "speaker_role"] = "audience_user"
v2.loc[creator_or_brand_flag, "speaker_role"] = "creator_or_brand"

# Generic channel/video praise without a substantive EV-bus opinion.
GENERIC_PRAISE_TERMS = {
    "analysis", "information", "knowledge", "perspective", "explainer", "video", "upload",
    "content", "yawn", "english", "strategy",
}
EV_OPINION_TERMS = {
    "battery", "charging", "charger", "range", "fare", "service", "refund", "seat", "ride",
    "comfort", "breakdown", "maintenance", "driver", "route", "depot", "reliability", "cost",
    "price", "investment", "tata", "jbm", "olectra", "nuego", "pmi", "eka", "switch",
}
def ascii_tokens(text: str) -> set[str]:
    return set(re.findall(r"[a-z]+", str(text).casefold()))

token_sets = v2["raw_text"].map(ascii_tokens)
generic_praise_flag = (
    (word_count <= 12)
    & (v2["entity_name_canonical"] == "General/Industry")
    & token_sets.map(lambda tokens: bool(tokens & GENERIC_PRAISE_TERMS) and not bool(tokens & EV_OPINION_TERMS))
)

# Build connected components using only the approved near_duplicate_partner links.
parent = {cid: cid for cid in v2["comment_id"]}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

id_set = set(v2["comment_id"])
edge_count = 0
for row in v2.itertuples(index=False):
    partner = str(row.near_duplicate_partner).split("|", 1)[0].strip()
    if partner and partner in id_set and partner != row.comment_id:
        union(row.comment_id, partner)
        edge_count += 1

components = defaultdict(list)
for cid in v2["comment_id"]:
    components[find(cid)].append(cid)
clusters = [sorted(members) for members in components.values() if len(members) > 1]
clusters.sort(key=lambda members: members[0])

cluster_by_id = {}
representative_ids = set()
row_by_id = v2.set_index("comment_id", drop=False)
id_to_pos = {cid: pos for pos, cid in enumerate(v2["comment_id"])}

for members in clusters:
    cluster_hash = hashlib.sha256("|".join(members).encode("utf-8")).hexdigest()[:14]
    cluster_id = f"ndc_{cluster_hash}"
    for cid in members:
        cluster_by_id[cid] = cluster_id

    def representative_rank(cid):
        row = row_by_id.loc[cid]
        pos = id_to_pos[cid]
        wrapped = str(row.raw_text).casefold().startswith(("user says:", "i observed that"))
        unusable = bool(row.exclusion_reason.strip()) or bool(flag_off_topic.iloc[pos])
        creator = bool(creator_or_brand_flag.iloc[pos])
        missing_lineage = not bool(str(row.url).strip() or str(row.date).strip())
        length = len(str(row.raw_text))
        return (wrapped, unusable, creator, missing_lineage, -length, cid)

    representative_ids.add(min(members, key=representative_rank))

v2["near_duplicate_cluster_id"] = v2["comment_id"].map(cluster_by_id).fillna("")
v2["near_duplicate_representative"] = v2["comment_id"].isin(representative_ids)
duplicate_copy = v2["near_duplicate_cluster_id"].ne("") & ~v2["near_duplicate_representative"]

# Regenerate topic text: no suffix stemming. Use local WordNet only when already available.
DOMAIN_TERMS = {
    "ev", "electric", "ebus", "ebuses", "bus", "buses", "battery", "batteries", "charging",
    "charger", "chargers", "range", "depot", "fleet", "tender", "procurement", "operator",
    "oem", "mobility", "service", "maintenance", "breakdown", "refund", "route", "fare",
    "passenger", "driver", "tata", "jbm", "vecv", "eicher", "olectra", "eka", "pmi",
    "switch", "nuego", "anthony",
}
HINGLISH_TERMS = {
    "aap", "aapko", "acha", "achha", "agar", "aur", "bahut", "bhai", "bhi", "hai", "hain",
    "ham", "hum", "ka", "kar", "karo", "ke", "ki", "ko", "kya", "lekin", "mat", "mein",
    "mera", "nahi", "nhi", "paisa", "par", "raha", "rahi", "sahi", "se", "sirf", "tha",
    "toh", "wala", "wali", "yaar", "ye", "yeh",
}
STOPWORDS = set("a about above after again against all am an and any are as at be because been before being below between both but by can could did do does doing down during each few for from further had has have having he her here hers herself him himself his how i if in into is it its itself just me more most my myself no nor not now of off on once only or other our ours ourselves out over own same she should so some such than that the their theirs them themselves then there these they this those through to too under until up very was we were what when where which while who whom why will with you your yours yourself yourselves".split())
STOPWORDS -= DOMAIN_TERMS
STOPWORDS -= HINGLISH_TERMS
URL_RE_V2 = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)

wordnet_lemmatizer = None
try:
    from nltk.corpus import wordnet
    from nltk.stem import WordNetLemmatizer
    _ = wordnet.synsets("bus")
    wordnet_lemmatizer = WordNetLemmatizer()
except (ImportError, LookupError):
    wordnet_lemmatizer = None

if wordnet_lemmatizer is not None:
    topic_method = "wordnet_verb_then_noun_no_stemming_v2"
    def maybe_lemma(token: str) -> str:
        if token in DOMAIN_TERMS or token in HINGLISH_TERMS:
            return token
        return wordnet_lemmatizer.lemmatize(wordnet_lemmatizer.lemmatize(token, pos="v"), pos="n")
else:
    topic_method = "cleaned_full_words_no_stemming_v2"
    def maybe_lemma(token: str) -> str:
        return token

def topic_clean_v2(text: str) -> str:
    text = URL_RE_V2.sub(" ", str(text).casefold())
    text = "".join(ch if ch.isalpha() or ch.isspace() else " " for ch in text)
    tokens = [t for t in re.findall(r"[^\W\d_]+", text, flags=re.UNICODE) if len(t) > 1]
    tokens = [maybe_lemma(t) for t in tokens if t not in STOPWORDS]
    return " ".join(tokens)

v2["topic_model_text"] = v2["raw_text"].map(topic_clean_v2)
v2["topic_preprocessing_version"] = topic_method

# Separate eligibility rules.
base_unusable = v2["exclusion_reason"].str.strip().ne("") | flag_off_topic
sentiment_eligible = ~base_unusable & ~creator_or_brand_flag & ~duplicate_copy & ~generic_praise_flag
topic_model_eligible = sentiment_eligible & ~flag_short & v2["topic_model_text"].str.strip().ne("")

v2["sentiment_eligible"] = sentiment_eligible
v2["topic_model_eligible"] = topic_model_eligible

def eligibility_reasons(i: int) -> str:
    reasons = []
    if v2.at[i, "exclusion_reason"].strip(): reasons.append(v2.at[i, "exclusion_reason"].strip())
    if bool(flag_off_topic.iloc[i]): reasons.append("clearly_off_topic")
    if bool(creator_or_brand_flag.iloc[i]): reasons.append("creator_or_brand")
    if bool(duplicate_copy.iloc[i]): reasons.append("near_duplicate_copy")
    if bool(generic_praise_flag.iloc[i]): reasons.append("generic_channel_praise")
    if bool(flag_short.iloc[i]) and bool(sentiment_eligible.iloc[i]): reasons.append("very_short_topic_ineligible")
    if not reasons:
        reasons.append("eligible_hinglish_retained" if bool(flag_hinglish.iloc[i]) else "eligible")
    return ";".join(dict.fromkeys(reasons))

v2["eligibility_reason"] = [eligibility_reasons(i) for i in range(len(v2))]
v2["manual_review_required"] = (
    creator_or_brand_flag | flag_off_topic | flag_promo | flag_short
    | v2["near_duplicate_representative"] | generic_praise_flag
    | (v2["speaker_role"] == "unknown")
)

# Human-review file: mandatory flags/representatives plus one deterministic row per stratum.
selection_reasons = defaultdict(set)
def add_selection(mask: pd.Series, reason: str):
    for cid in v2.loc[mask, "comment_id"]:
        selection_reasons[cid].add(reason)

add_selection(creator_or_brand_flag, "creator_or_brand_flag")
add_selection(flag_off_topic, "possible_off_topic_flag")
add_selection(flag_promo, "promotional_flag")
add_selection(flag_short, "very_short_row")
add_selection(v2["near_duplicate_representative"], "duplicate_cluster_representative")
add_selection(generic_praise_flag, "generic_channel_praise")
add_selection(v2["speaker_role"].eq("unknown"), "unknown_speaker_role")

strata = ["entity_type", "oem_group", "platform_canonical"]
for _, group in v2.groupby(strata, dropna=False, sort=True):
    candidates = group.sort_values("comment_id")
    chosen = candidates.sample(n=1, random_state=PATCH_SEED).iloc[0]
    selection_reasons[chosen.comment_id].add("stratified_sample")

review_ids = sorted(selection_reasons)
review_columns = [
    "comment_id", "near_duplicate_cluster_id", "near_duplicate_representative",
    "platform_original", "platform_canonical", "source", "url", "date", "provenance",
    "entity_name_original", "entity_name_canonical", "entity_type", "oem_group",
    "speaker_role", "raw_text", "quality_flag", "eligibility_reason",
    "sentiment_eligible", "topic_model_eligible", "manual_review_required",
]
manual_review = v2.set_index("comment_id").loc[review_ids, review_columns[1:]].reset_index()
manual_review["review_selection_reason"] = manual_review["comment_id"].map(
    lambda cid: ";".join(sorted(selection_reasons[cid]))
)
for column in [
    "reviewer_decision", "corrected_speaker_role", "corrected_sentiment_eligible",
    "corrected_topic_model_eligible", "reviewer_note",
]:
    manual_review[column] = ""

# Summary metrics and reason counts.
summary_rows = [
    ("headline", "retained_rows", len(v2), "All approved v1 rows retained"),
    ("headline", "duplicate_clusters", len(clusters), "Connected components from near_duplicate_partner"),
    ("headline", "duplicate_representatives", int(v2["near_duplicate_representative"].sum()), "Exactly one per duplicate cluster"),
    ("headline", "duplicate_copies", int(duplicate_copy.sum()), "Cluster members other than representative"),
    ("headline", "sentiment_eligible_rows", int(v2["sentiment_eligible"].sum()), "Rows eligible for later sentiment analysis"),
    ("headline", "topic_model_eligible_rows", int(v2["topic_model_eligible"].sum()), "Rows eligible for later topic modelling"),
    ("headline", "creator_or_brand_flags", int(creator_or_brand_flag.sum()), "Conservative obvious creator/brand rule"),
    ("headline", "off_topic_flags", int(flag_off_topic.sum()), "Inherited possible off-topic flag"),
    ("headline", "manual_review_rows", len(manual_review), "Union of required flags and stratified sample"),
]
for role, count in v2["speaker_role"].value_counts().sort_index().items():
    summary_rows.append(("speaker_role", str(role), int(count), "Speaker-role distribution"))
for platform, count in v2["platform_canonical"].value_counts().sort_index().items():
    summary_rows.append(("platform", str(platform), int(count), "Canonical platform distribution"))
for reason, count in v2["eligibility_reason"].value_counts().sort_index().items():
    summary_rows.append(("eligibility_reason", str(reason), int(count), "Non-exclusive logic encoded as combined row reason"))

eligibility_summary = pd.DataFrame(summary_rows, columns=["section", "metric", "row_count", "definition"])
eligibility_summary["percent_of_548"] = (eligibility_summary["row_count"] / len(v2) * 100).round(2)

# Preserve v1 columns first, then append patch fields.
PATCH_COLUMNS = [
    "platform_original", "platform_canonical", "near_duplicate_cluster_id",
    "near_duplicate_representative", "speaker_role", "sentiment_eligible",
    "topic_model_eligible", "eligibility_reason", "manual_review_required",
]
v2 = v2[v1_columns + PATCH_COLUMNS]

v2.to_csv(V2_PATH, index=False)
manual_review.to_csv(REVIEW_PATH, index=False)
eligibility_summary.to_csv(SUMMARY_PATH, index=False)

report = f"""# Preprocessing Patch Report

## Scope

Targeted final data-quality patch applied to the approved 548-row processed corpus. All v1 rows and columns are retained. No original or interim file was modified. No synthetic data, internet browsing, lifecycle classification, VADER scoring, or LDA/topic modelling was used.

## Results

- Retained rows: **{len(v2)}**
- Duplicate clusters: **{len(clusters)}**
- Duplicate representatives: **{int(v2["near_duplicate_representative"].sum())}**
- Duplicate copies: **{int(duplicate_copy.sum())}**
- Sentiment-eligible rows: **{int(v2["sentiment_eligible"].sum())}**
- Topic-model-eligible rows: **{int(v2["topic_model_eligible"].sum())}**
- Creator/brand flags: **{int(creator_or_brand_flag.sum())}**
- Off-topic flags: **{int(flag_off_topic.sum())}**
- Manual-review rows: **{len(manual_review)}**

## Near-duplicate handling

The existing `near_duplicate_partner` links were treated as undirected graph edges. Each connected component of two or more rows received a deterministic `near_duplicate_cluster_id` derived from its sorted comment IDs. Exactly one representative was selected per cluster, preferring non-wrapper text, usable/audience text, available URL/date lineage, and richer text; comment ID breaks ties. No row was deleted. Non-representative cluster copies are ineligible for sentiment and topic modelling.

## Eligibility rules

- Blank, URL-only, emoji-only, inherited possible-off-topic rows, and obvious creator/brand content are ineligible for both uses.
- Near-duplicate representatives remain potentially eligible; duplicate copies are ineligible for both uses.
- Generic channel/video praise without a substantive EV-bus opinion is ineligible for both uses.
- Very-short but otherwise meaningful rows remain potentially sentiment eligible and are topic-model ineligible.
- Hinglish/code-switched rows are not automatically excluded.
- `eligibility_reason` records the applied rule; `manual_review_required` marks uncertain or high-risk cases.

## Speaker role

Controlled roles are `audience_user`, `creator_or_brand`, `media_or_research`, and `unknown`. Classification is conservative. Obvious channel-host language and likely OEM Instagram captions are flagged as creator/brand; ambiguous Instagram material remains unknown or audience only where user voice is explicit.

## Platform normalisation

The existing `platform` field is unchanged and copied to `platform_original`. `platform_canonical` consolidates variants into YouTube, Instagram, Reddit, Facebook, LinkedIn, Forum, Review Platform, News/Media, or Mixed/Unknown.

## Topic text regeneration

`topic_model_text` was regenerated from `raw_text`: lowercased; URLs, punctuation, standalone numbers, and explicit English stopwords removed; EV-bus domain and Hinglish terms protected. Aggressive suffix rules are not used. Method for this run: **{topic_method}**. If local WordNet data is unavailable, cleaned full words are retained without stemming.

## Manual review

The review file contains every creator/brand flag, possible off-topic flag, promotional flag, very-short row, duplicate-cluster representative, generic channel-praise flag, unknown-speaker row, plus a deterministic stratified sample across entity type, OEM group, and canonical platform. Human-decision fields are blank by design.

## Limitations

- Near-duplicate links inherit the v1 heuristic and may join paraphrases that require human confirmation.
- Speaker role is inferred without original author/channel-owner fields.
- `possible_off_topic_content` is an inherited heuristic flag and remains subject to review.
- Platform and provenance fields remain derived from an imbalanced mixed-provenance corpus.
- Eligibility is a preparation decision, not an analytical result or human-validated ground truth.
"""
REPORT_PATH.write_text(report, encoding="utf-8")

# Integrity assertions.
reloaded = pd.read_csv(V2_PATH, dtype=str, keep_default_na=False)
assert len(reloaded) == 548
assert list(reloaded.columns[:len(v1_columns)]) == v1_columns
for column in [c for c in v1_columns if c not in {"topic_model_text", "topic_preprocessing_version"}]:
    assert reloaded[column].tolist() == v1[column].tolist(), f"Changed v1 field: {column}"
assert reloaded["comment_id"].is_unique
assert int(reloaded["near_duplicate_representative"].eq("True").sum()) == len(clusters)
assert reloaded.loc[reloaded["near_duplicate_cluster_id"].ne(""), "near_duplicate_cluster_id"].nunique() == len(clusters)
assert not ((reloaded["near_duplicate_cluster_id"].ne("")) & (reloaded["near_duplicate_representative"] == "False") & ((reloaded["sentiment_eligible"] == "True") | (reloaded["topic_model_eligible"] == "True"))).any()
assert not (reloaded["speaker_role"].eq("creator_or_brand") & ((reloaded["sentiment_eligible"] == "True") | (reloaded["topic_model_eligible"] == "True"))).any()
assert set(reloaded["speaker_role"]) <= {"audience_user", "creator_or_brand", "media_or_research", "unknown"}
assert set(reloaded["platform_canonical"]) <= {"YouTube", "Instagram", "Reddit", "Facebook", "LinkedIn", "Forum", "Review Platform", "News/Media", "Mixed/Unknown"}
assert manual_review[["reviewer_decision", "corrected_speaker_role", "corrected_sentiment_eligible", "corrected_topic_model_eligible", "reviewer_note"]].eq("").all().all()

print(f"Saved {V2_PATH.relative_to(PROJECT_ROOT)}")
print(f"Duplicate clusters: {len(clusters)}; representatives: {len(representative_ids)}")
print(f"Sentiment eligible: {int(sentiment_eligible.sum())}; topic eligible: {int(topic_model_eligible.sum())}")
print(f"Manual review rows: {len(manual_review)}; topic method: {topic_method}")